# Creating Specialties Dimensions

This script creates one row per NPI per year with the most recent specialty information within each year. Specialties can change across years for the same NPI.

This script relies on the DMEPOS Referring Provider and General Payments for specialty information. 

This script requires /dsa/groups/casestudycf25/team02/general_payments_providers_clean.csv and /dsa/groups/casestudycf25/team02/DMEPOS_rfrr_clean.csv.

Data Cleaning
- Keep one row per NPI + year from the DMEPOS dataset, prioritizing NPPES sourced and most recent data within year
- Keep one row per NPI + year from General Payments provider csv
- Merge and prioritize: NPPES source > Claim source > Payments source

It returns ("dmepos_specialties_unique.csv", "gp_specialties_unique.csv") as interim CSVs.

It *requires* a manually created CSV called specialty_crosswalk_gpt_backfill.csv
This manual CSV was created by using the outputs of the dmepos_specialties_unique and the gp_specialties_unique CSVs for manual deduplication and backfill of specialty hierarchy information. ChatGPT was used to fill out the groups and then manual validation was applied to the groupings. This is then merged to the one_specialty_per_npi_final by the og_specialty_name to get the hierarchy decriptions. 

It returns the final one row per NPI per year with specialty information dataset: /dsa/groups/casestudycf25/team02/one_specialty_per_npi.csv 

In [1]:
import pandas as pd

### Using the DMEPOS Referring Providers Dataset

In [2]:
dme_year_npi = pd.read_csv('/dsa/groups/casestudycf25/team02/silver/dmepos_rfrr.csv',dtype={'rfrg_prvdr_state_fips':str,'rfrg_prvdr_zip5':str}) # ensure Rfrg_Prvdr_State_FIPS & Rfrg_Prvdr_Zip5 are imported as str


In [3]:
dme_year_npi.columns

Index(['rfrg_npi', 'rfrg_prvdr_last_name_org', 'rfrg_prvdr_first_name',
       'rfrg_prvdr_mi', 'rfrg_prvdr_crdntls', 'rfrg_prvdr_ent_cd',
       'rfrg_prvdr_st1', 'rfrg_prvdr_st2', 'rfrg_prvdr_city',
       'rfrg_prvdr_state_abrvtn', 'rfrg_prvdr_state_fips', 'rfrg_prvdr_zip5',
       'rfrg_prvdr_ruca', 'rfrg_prvdr_ruca_desc', 'rfrg_prvdr_cntry',
       'rfrg_prvdr_spclty_desc', 'rfrg_prvdr_spclty_srce', 'tot_suplrs',
       'tot_suplr_hcpcs_cds', 'tot_suplr_benes', 'tot_suplr_clms',
       'tot_suplr_srvcs', 'suplr_sbmtd_chrgs', 'suplr_mdcr_alowd_amt',
       'suplr_mdcr_pymt_amt', 'suplr_mdcr_stdzd_pymt_amt', 'dme_sprsn_ind',
       'dme_tot_suplrs', 'dme_tot_suplr_hcpcs_cds', 'dme_tot_suplr_benes',
       'dme_tot_suplr_clms', 'dme_tot_suplr_srvcs', 'dme_suplr_sbmtd_chrgs',
       'dme_suplr_mdcr_alowd_amt', 'dme_suplr_mdcr_pymt_amt',
       'dme_suplr_mdcr_stdzd_pymt_amt', 'pos_sprsn_ind', 'pos_tot_suplrs',
       'pos_tot_suplr_hcpcs_cds', 'pos_tot_suplr_benes', 'pos_tot_suplr_clm

In [4]:
dme_year_npi['rfrg_prvdr_spclty_srce'] = dme_year_npi['rfrg_prvdr_spclty_srce'].fillna('Claim-Specialty')
dme_year_npi.loc[dme_year_npi['rfrg_prvdr_spclty_srce'] == '', 'rfrg_prvdr_spclty_srce'] = 'Claim-Specialty'

In [5]:
dme_year_npi["rfrg_prvdr_spclty_srce"].unique()

array(['Claim-Specialty', 'NPPES-Specialty', 'NPPES-Taxonomy'],
      dtype=object)

In [6]:
dme_specialties = dme_year_npi[["rfrg_prvdr_spclty_desc"]].copy()
dme_specialties.drop_duplicates(inplace = True, keep="first")
dme_specialties

,rfrg_prvdr_spclty_desc
0,Internal Medicine
1,General Surgery
2,Family Practice
4,Urology
6,Cardiology
...,...
864619,Clinical Neuropsychologist
888244,Home Health
896675,Rehabilitation Counselor
925565,Integrative Medicine


In [7]:
dme_specialties.to_csv("dmepos_specialties_unique.csv")

In [8]:
dme_specialties_npi = dme_year_npi[["rfrg_npi","rfrg_prvdr_spclty_desc", "rfrg_prvdr_spclty_srce" , "year"]].copy()

In [9]:
dme_specialties_npi

,rfrg_npi,rfrg_prvdr_spclty_desc,rfrg_prvdr_spclty_srce,year
0,1003000126,Internal Medicine,Claim-Specialty,2021
1,1003000480,General Surgery,Claim-Specialty,2021
2,1003000522,Family Practice,NPPES-Specialty,2021
3,1003000530,Internal Medicine,Claim-Specialty,2021
4,1003000597,Urology,Claim-Specialty,2021
...,...,...,...,...
1157248,1992998645,Internal Medicine,Claim-Specialty,2023
1157249,1992998736,Family Practice,Claim-Specialty,2023
1157250,1992999031,Hematology-Oncology,Claim-Specialty,2023
1157251,1992999122,Internal Medicine,Claim-Specialty,2023


In [10]:
# Filter by NPPES Specialty derived rows
dme_specialties_nppes = dme_specialties_npi[
    (dme_specialties_npi["rfrg_prvdr_spclty_srce"] == "NPPES-Specialty") |
    (dme_specialties_npi["rfrg_prvdr_spclty_srce"] == "NPPES-Taxonomy")
]
dme_specialties_nppes = dme_specialties_nppes.sort_values('year', ascending=False)

# Keep the first and most recent specialty information per NPI + year
dme_specialties_nppes.drop_duplicates(subset = ["rfrg_npi", "year"] , inplace = True, keep="first")

In [11]:
# Filter by Claim Specialty derived rows
dme_specialties_claims = dme_specialties_npi[dme_specialties_npi["rfrg_prvdr_spclty_srce"] == "Claim-Specialty"]
dme_specialties_claims = dme_specialties_claims.sort_values('year', ascending=False)

# Keep the first and most recent specialty information per NPI + year
dme_specialties_claims.drop_duplicates(subset = ["rfrg_npi", "year"] , inplace = True, keep="first")

In [12]:
# Append the specialties subsets, with the NPPES sourced rows first
temp_append = pd.concat([dme_specialties_nppes, dme_specialties_claims], ignore_index=True)

In [13]:
# Sort it by the source to prioritize NPPES specialty information first
dme_specialties_npi_sorted = temp_append.sort_values('rfrg_prvdr_spclty_srce', ascending=False)

# Keep only one row per NPI + year from the DMEPOS dataset, prioritizing NPPES sourced and most recent info 
dme_specialties_npi_sorted.drop_duplicates(subset = ["rfrg_npi", "year"] , inplace = True, keep="first")

In [14]:
dme_specialties_npi_sorted

,rfrg_npi,rfrg_prvdr_spclty_desc,rfrg_prvdr_spclty_srce,year
55270,1043273253,Internal Medicine,NPPES-Taxonomy,2022
65806,1275631657,Specialist,NPPES-Taxonomy,2022
18074,1821084567,Internal Medicine,NPPES-Taxonomy,2023
18072,1821080375,Internal Medicine,NPPES-Taxonomy,2023
18069,1821219569,Obstetrics & Gynecology,NPPES-Taxonomy,2023
...,...,...,...,...
453498,1669960704,Geriatric Medicine,Claim-Specialty,2023
453499,1669960951,Internal Medicine,Claim-Specialty,2023
453500,1669961215,Family Practice,Claim-Specialty,2023
453501,1669961579,Physician Assistant,Claim-Specialty,2023


In [15]:
del dme_specialties_npi , dme_specialties_nppes, dme_specialties_claims

### General Payments Specialties

In [16]:
gp_providers = pd.read_csv("/dsa/groups/casestudycf25/team02/silver/general_payments_providers_clean.csv")

### Data Quality Check: Null Specialty Values

In [17]:
# Check null specialty values in DMEPOS dataset
print("NULL SPECIALTY CHECK - DMEPOS DATASET")
print("=" * 80)
total_dmepos_rows = len(dme_year_npi)
null_specialty_dmepos = dme_year_npi['rfrg_prvdr_spclty_desc'].isna().sum()
pct_null_dmepos = (null_specialty_dmepos / total_dmepos_rows) * 100

print(f"Total rows: {total_dmepos_rows:,}")
print(f"Rows with null specialty: {null_specialty_dmepos:,}")
print(f"Percentage null: {pct_null_dmepos:.2f}%")

# Check unique NPI-year combinations with null specialty
dmepos_null_npi_years = dme_year_npi[dme_year_npi['rfrg_prvdr_spclty_desc'].isna()][['rfrg_npi', 'year']].drop_duplicates()
print(f"Unique NPI-year combinations with null specialty: {len(dmepos_null_npi_years):,}")

NULL SPECIALTY CHECK - DMEPOS DATASET
Total rows: 1,157,253
Rows with null specialty: 0
Percentage null: 0.00%
Unique NPI-year combinations with null specialty: 0


In [18]:
# Check null specialty values in General Payments dataset
print("\nNULL SPECIALTY CHECK - GENERAL PAYMENTS DATASET")
print("=" * 80)
total_gp_rows = len(gp_providers)
null_specialty_gp = gp_providers['specialty_lvl2'].isna().sum()
pct_null_gp = (null_specialty_gp / total_gp_rows) * 100

print(f"Total rows: {total_gp_rows:,}")
print(f"Rows with null specialty: {null_specialty_gp:,}")
print(f"Percentage null: {pct_null_gp:.2f}%")

# Check unique NPI-year combinations with null specialty
gp_null_npi_years = gp_providers[gp_providers['specialty_lvl2'].isna()][['covered_recipient_npi', 'program_year']].drop_duplicates()
print(f"Unique NPI-year combinations with null specialty: {len(gp_null_npi_years):,}")


NULL SPECIALTY CHECK - GENERAL PAYMENTS DATASET
Total rows: 266,401
Rows with null specialty: 0
Percentage null: 0.00%
Unique NPI-year combinations with null specialty: 0


In [19]:
# Summary of null specialty impact
print("\nSUMMARY")
print("=" * 80)
print(f"DMEPOS: {pct_null_dmepos:.2f}% of rows have null specialty")
print(f"General Payments: {pct_null_gp:.2f}% of rows have null specialty")
print("\nThese null values will be excluded during processing and will not appear")
print("in the final dataset after the crosswalk merge.")


SUMMARY
DMEPOS: 0.00% of rows have null specialty
General Payments: 0.00% of rows have null specialty

These null values will be excluded during processing and will not appear
in the final dataset after the crosswalk merge.


In [20]:
gp_providers.head()

,Unnamed: 0,covered_recipient_profile_id,covered_recipient_npi,covered_recipient_type,covered_recipient_primary_type_1,covered_recipient_specialty_1,specialty_lvl1,specialty_lvl2,specialty_lvl3,covered_recipient_license_state_code1,...,covered_recipient_license_state_code4,covered_recipient_license_state_code5,recipient_primary_business_street_address_line1,recipient_primary_business_street_address_line2,recipient_city,recipient_state,recipient_zip_code,recipient_country,recipient_postal_code,program_year
0,7,11452165,1750964185,Covered Recipient Physician,Medical Doctor,Allopathic & Osteopathic Physicians|Orthopaedi...,Allopathic & Osteopathic Physicians,Orthopaedic Surgery,NaN,IL,...,NaN,NaN,600 S PAULINA ST,NaN,CHICAGO,IL,60612,United States,NaN,2023
1,19,745711,1417944091,Covered Recipient Physician,Medical Doctor,Allopathic & Osteopathic Physicians|Orthopaedi...,Allopathic & Osteopathic Physicians,Orthopaedic Surgery,Foot and Ankle Surgery,IL,...,NaN,NaN,1301 S KOKE MILL RD,NaN,SPRINGFIELD,IL,62711,United States,NaN,2023
2,23,8757794,1699208850,Covered Recipient Physician,Doctor of Podiatric Medicine,Podiatric Medicine & Surgery Service Providers...,Podiatric Medicine & Surgery Service Providers,Podiatrist,Foot & Ankle Surgery,NM,...,NaN,NaN,693 PRESIDENT PL STE 103,NaN,SMYRNA,TN,37167,United States,NaN,2023
3,28,50178,1124052717,Covered Recipient Physician,Doctor of Podiatric Medicine,Podiatric Medicine & Surgery Service Providers...,Podiatric Medicine & Surgery Service Providers,Podiatrist,Foot & Ankle Surgery,IL,...,NaN,NaN,3471 GREEN BAY RD,NaN,NORTH CHICAGO,IL,60064,United States,NaN,2023
4,36,154326,1386832855,Covered Recipient Physician,Medical Doctor,Allopathic & Osteopathic Physicians|Orthopaedi...,Allopathic & Osteopathic Physicians,Orthopaedic Surgery,NaN,IL,...,NaN,NaN,2905 N MAIN ST,SUITE G,DECATUR,IL,62526,United States,NaN,2023


In [21]:
gp_specialties = gp_providers[["covered_recipient_primary_type_1", "specialty_lvl1", "specialty_lvl2"]].copy()
gp_specialties.drop_duplicates(subset = 'specialty_lvl2', inplace = True, keep="first")
gp_specialties

,covered_recipient_primary_type_1,specialty_lvl1,specialty_lvl2
0,Medical Doctor,Allopathic & Osteopathic Physicians,Orthopaedic Surgery
2,Doctor of Podiatric Medicine,Podiatric Medicine & Surgery Service Providers,Podiatrist
7,Medical Doctor,Allopathic & Osteopathic Physicians,Anesthesiology
12,Medical Doctor,Allopathic & Osteopathic Physicians,Surgery
16,Medical Doctor,Allopathic & Osteopathic Physicians,Otolaryngology
19,Medical Doctor,Allopathic & Osteopathic Physicians,Emergency Medicine
48,Medical Doctor,Allopathic & Osteopathic Physicians,Radiology
70,Medical Doctor,Allopathic & Osteopathic Physicians,Internal Medicine
76,Doctor of Osteopathy,Allopathic & Osteopathic Physicians,Family Medicine
95,Medical Doctor,Allopathic & Osteopathic Physicians,Pediatrics


In [22]:
gp_specialties.to_csv("gp_specialties_unique.csv")

In [23]:
gp_specialties_npi = gp_providers[["covered_recipient_npi","specialty_lvl2", "program_year"]].copy()

In [24]:
gp_specialties_npi = gp_specialties_npi.sort_values('program_year', ascending=False)

# Keep most recent program year specialty information per NPI + year
gp_specialties_npi.drop_duplicates(subset = ["covered_recipient_npi", "program_year"] , inplace = True, keep="first")

In [25]:
gp_specialties_npi["rfrg_prvdr_spclty_srce"] = "Payments-Specialty"

In [26]:
gp_specialties_npi

,covered_recipient_npi,specialty_lvl2,program_year,rfrg_prvdr_spclty_srce
0,1750964185,Orthopaedic Surgery,2023,Payments-Specialty
82721,1912067661,Optometrist,2023,Payments-Specialty
82732,1386872679,Optometrist,2023,Payments-Specialty
82731,1407087885,Optometrist,2023,Payments-Specialty
82730,1427161611,Optometrist,2023,Payments-Specialty
...,...,...,...,...
138834,1669689949,Orthopaedic Surgery,2021,Payments-Specialty
138815,1942259445,Family Medicine,2021,Payments-Specialty
138827,1194760041,Internal Medicine,2021,Payments-Specialty
138825,1093248668,Internal Medicine,2021,Payments-Specialty


In [27]:
gp_specialties_npi = gp_specialties_npi.rename(columns = {
        "covered_recipient_npi": "rfrg_npi",
        "specialty_lvl2": "rfrg_prvdr_spclty_desc",
        "program_year": "year",
        "Rfrg_Prvdr_Spclty_Srce": "rfrg_prvdr_spclty_srce"})

In [28]:
gp_specialties_npi

,rfrg_npi,rfrg_prvdr_spclty_desc,year,rfrg_prvdr_spclty_srce
0,1750964185,Orthopaedic Surgery,2023,Payments-Specialty
82721,1912067661,Optometrist,2023,Payments-Specialty
82732,1386872679,Optometrist,2023,Payments-Specialty
82731,1407087885,Optometrist,2023,Payments-Specialty
82730,1427161611,Optometrist,2023,Payments-Specialty
...,...,...,...,...
138834,1669689949,Orthopaedic Surgery,2021,Payments-Specialty
138815,1942259445,Family Medicine,2021,Payments-Specialty
138827,1194760041,Internal Medicine,2021,Payments-Specialty
138825,1093248668,Internal Medicine,2021,Payments-Specialty


In [29]:
one_specialty_per_npi = pd.concat([dme_specialties_npi_sorted, gp_specialties_npi], ignore_index=True)

In [30]:
one_specialty_per_npi.shape

(1316004, 4)

In [31]:
one_specialty_per_npi.head()

,rfrg_npi,rfrg_prvdr_spclty_desc,rfrg_prvdr_spclty_srce,year
0,1043273253,Internal Medicine,NPPES-Taxonomy,2022
1,1275631657,Specialist,NPPES-Taxonomy,2022
2,1821084567,Internal Medicine,NPPES-Taxonomy,2023
3,1821080375,Internal Medicine,NPPES-Taxonomy,2023
4,1821219569,Obstetrics & Gynecology,NPPES-Taxonomy,2023


In [32]:
one_specialty_per_npi.drop_duplicates(subset = ["rfrg_npi", "year"] , inplace = True, keep="first")

In [33]:
one_specialty_per_npi.shape

(1265154, 4)

### Specialty Crosswalk

Merge the manually created specialty_crosswalk_gpt_backfill.csv

Directions on how to create the specialty_crosswalk_gpt_backfill.csv:
- This manual CSV was created by using the outputs of this script (=dmepos_specialties_unique and the gp_specialties_unique CSVs). It was manually deduplicated and ChatGPT was used to fill out the group levels. Then manual validation was applied to the new backfilled information. 

This is then merged to the one_specialty_per_npi_final by the og_specialty_name and Rfrg_Prvdr_Spclty_Desc to get the hierarchy decriptions. 

In [34]:
specialty_crosswalk = pd.read_csv("/dsa/groups/casestudycf25/team02/bronze/specialty_crosswalk_gpt_backfill.csv")

In [35]:
specialty_crosswalk.head()

,specialty_type,specialty_lvl1,specialty,og_specialty_name
0,Chiropractor,Chiropractic Providers,Chiropractor,Chiropractic
1,Chiropractor,Chiropractic Providers,Chiropractor,Chiropractor
2,Doctor of Dentistry,Dental Providers,Advanced Practice Dental Therapist,Advanced Practice Dental Therapist
3,Doctor of Dentistry,Dental Providers,Dental Assistant,Dental Assistant
4,Doctor of Dentistry,Dental Providers,Dental Hygienist,Dental Hygienist


In [36]:
specialty_crosswalk.to_csv("specialty_crosswalk.csv")

In [37]:
one_specialty_per_npi_final = one_specialty_per_npi.merge(
    specialty_crosswalk,
    left_on = "rfrg_prvdr_spclty_desc",
    right_on = "og_specialty_name",
    how="inner",          
    validate="many_to_one" 
)
one_specialty_per_npi_final.shape

(1265154, 8)

In [38]:
one_specialty_per_npi_final.drop(columns = ["rfrg_prvdr_spclty_desc"], inplace =True)

In [39]:
one_specialty_per_npi_final

,rfrg_npi,rfrg_prvdr_spclty_srce,year,specialty_type,specialty_lvl1,specialty,og_specialty_name
0,1043273253,NPPES-Taxonomy,2022,Medical Doctor,Allopathic & Osteopathic Physicians,Internal Medicine,Internal Medicine
1,1275631657,NPPES-Taxonomy,2022,Other,Other Service Providers,Specialist,Specialist
2,1821084567,NPPES-Taxonomy,2023,Medical Doctor,Allopathic & Osteopathic Physicians,Internal Medicine,Internal Medicine
3,1821080375,NPPES-Taxonomy,2023,Medical Doctor,Allopathic & Osteopathic Physicians,Internal Medicine,Internal Medicine
4,1821219569,NPPES-Taxonomy,2023,Medical Doctor,Allopathic & Osteopathic Physicians,Obstetrics and Gynecology,Obstetrics & Gynecology
...,...,...,...,...,...,...,...
1265149,1891746319,Payments-Specialty,2021,Medical Doctor,Allopathic & Osteopathic Physicians,Anesthesiology,Anesthesiology
1265150,1629429857,Payments-Specialty,2021,Doctor of Dentistry,Dental Providers,Dentist,Dentist
1265151,1780646976,Payments-Specialty,2021,Doctor of Dentistry,Dental Providers,Dentist,Dentist
1265152,1457754004,Payments-Specialty,2021,Doctor of Dentistry,Dental Providers,Dentist,Dentist


In [40]:
# Number of NaNs in the column 'covered_recipient_npi'
n_nans = one_specialty_per_npi_final['specialty'].isna().sum()
print("NaNs in specialty:", n_nans)
nan_percent = one_specialty_per_npi_final['specialty'].isna().mean() * 100
print(f"Percentage of NaNs in specialty: {nan_percent:.2f}%")

NaNs in specialty: 0
Percentage of NaNs in specialty: 0.00%


In [41]:
# Verify one specialty per NPI per year
npi_year_counts = one_specialty_per_npi_final.groupby(['rfrg_npi', 'year']).size()
max_per_npi_year = npi_year_counts.max()
print(f"\nMax specialties per NPI-year: {max_per_npi_year}")
if max_per_npi_year > 1:
    print("WARNING: Some NPI-year combinations have multiple specialties!")
    duplicates = npi_year_counts[npi_year_counts > 1]
    print(f"Number of duplicated NPI-year combinations: {len(duplicates)}")
else:
    print("Confirmed: Each NPI has exactly one specialty per year")


Max specialties per NPI-year: 1
Confirmed: Each NPI has exactly one specialty per year


### Data Validation Checks

In [42]:
#All NPIs from DMEPOS are in final dataset
print("\nCHECKING NPIs FROM DMEPOS DATASET")
print("-" * 80)
dme_npis = set(dme_year_npi['rfrg_npi'].dropna().unique())
final_npis = set(one_specialty_per_npi_final['rfrg_npi'].dropna().unique())

dme_npis_in_final = dme_npis.intersection(final_npis)
dme_npis_missing = dme_npis - final_npis

print(f"Total unique NPIs in DMEPOS source: {len(dme_npis):,}")
print(f"DMEPOS NPIs found in final dataset: {len(dme_npis_in_final):,}")
print(f"DMEPOS NPIs missing from final: {len(dme_npis_missing):,}")
print(f"Coverage: {100 * len(dme_npis_in_final) / len(dme_npis):.2f}%")

if len(dme_npis_missing) > 0:
    print(f"\n WARNING: {len(dme_npis_missing)} NPIs from DMEPOS are missing!")
    print(f"Sample missing NPIs: {list(dme_npis_missing)[:5]}")
else:
    print("\nAll DMEPOS NPIs are present in final dataset")


CHECKING NPIs FROM DMEPOS DATASET
--------------------------------------------------------------------------------
Total unique NPIs in DMEPOS source: 498,197
DMEPOS NPIs found in final dataset: 498,197
DMEPOS NPIs missing from final: 0
Coverage: 100.00%

All DMEPOS NPIs are present in final dataset


In [43]:
# All NPIs from General Payments are in final dataset
print("\nCHECKING NPIs FROM GENERAL PAYMENTS DATASET")
print("-" * 80)
gp_npis = set(gp_providers['covered_recipient_npi'].dropna().unique())

gp_npis_in_final = gp_npis.intersection(final_npis)
gp_npis_missing = gp_npis - final_npis

print(f"Total unique NPIs in General Payments source: {len(gp_npis):,}")
print(f"General Payments NPIs found in final dataset: {len(gp_npis_in_final):,}")
print(f"General Payments NPIs missing from final: {len(gp_npis_missing):,}")
print(f"Coverage: {100 * len(gp_npis_in_final) / len(gp_npis):.2f}%")

if len(gp_npis_missing) > 0:
    print(f"\nWARNING: {len(gp_npis_missing)} NPIs from General Payments are missing!")
    print(f"Sample missing NPIs: {list(gp_npis_missing)[:5]}")
else:
    print("\nAll General Payments NPIs are present in final dataset")


CHECKING NPIs FROM GENERAL PAYMENTS DATASET
--------------------------------------------------------------------------------
Total unique NPIs in General Payments source: 85,287
General Payments NPIs found in final dataset: 85,287
General Payments NPIs missing from final: 0
Coverage: 100.00%

All General Payments NPIs are present in final dataset


In [44]:
#All specialties from DMEPOS are in final dataset
print("\nCHECKING SPECIALTIES FROM DMEPOS DATASET")
print("-" * 80)
dme_specialties_set = set(dme_year_npi['rfrg_prvdr_spclty_desc'].dropna().unique())
final_specialties = set(one_specialty_per_npi_final['og_specialty_name'].dropna().unique())

dme_specialties_in_final = dme_specialties_set.intersection(final_specialties)
dme_specialties_missing = dme_specialties_set - final_specialties

print(f"Total unique specialties in DMEPOS source: {len(dme_specialties_set):,}")
print(f"DMEPOS specialties found in final dataset: {len(dme_specialties_in_final):,}")
print(f"DMEPOS specialties missing from final: {len(dme_specialties_missing):,}")
print(f"Coverage: {100 * len(dme_specialties_in_final) / len(dme_specialties_set):.2f}%")

if len(dme_specialties_missing) > 0:
    print(f"\nWARNING: {len(dme_specialties_missing)} specialties from DMEPOS are missing!")
    print(f"Missing specialties: {sorted(dme_specialties_missing)}")
else:
    print("\nAll DMEPOS specialties are present in final dataset")


CHECKING SPECIALTIES FROM DMEPOS DATASET
--------------------------------------------------------------------------------
Total unique specialties in DMEPOS source: 132
DMEPOS specialties found in final dataset: 132
DMEPOS specialties missing from final: 0
Coverage: 100.00%

All DMEPOS specialties are present in final dataset


In [45]:
#All specialties from General Payments are in final dataset
print("\nCHECKING SPECIALTIES FROM GENERAL PAYMENTS DATASET")
print("-" * 80)
gp_specialties_set = set(gp_providers['specialty_lvl2'].dropna().unique())
final_rfrg_desc = set(one_specialty_per_npi_final['og_specialty_name'].dropna().unique())

gp_specialties_in_final = gp_specialties_set.intersection(final_rfrg_desc)
gp_specialties_missing = gp_specialties_set - final_rfrg_desc

print(f"Total unique specialties in General Payments source: {len(gp_specialties_set):,}")
print(f"General Payments specialties found in final dataset: {len(gp_specialties_in_final):,}")
print(f"General Payments specialties missing from final: {len(gp_specialties_missing):,}")
print(f"Coverage: {100 * len(gp_specialties_in_final) / len(gp_specialties_set):.2f}%")

if len(gp_specialties_missing) > 0:
    print(f"\nWARNING: {len(gp_specialties_missing)} specialties from General Payments are missing!")
    print(f"Missing specialties: {sorted(gp_specialties_missing)}")
else:
    print("\nAll General Payments specialties are present in final dataset")


CHECKING SPECIALTIES FROM GENERAL PAYMENTS DATASET
--------------------------------------------------------------------------------
Total unique specialties in General Payments source: 46
General Payments specialties found in final dataset: 43
General Payments specialties missing from final: 3
Coverage: 93.48%

Missing specialties: ['Electrodiagnostic Medicine', 'Independent Medical Examiner', 'Technician/Technologist']


In [46]:
# Check 5: All NPI-Year combinations are preserved (with one specialty each)
print("\n5. CHECKING NPI-YEAR COMBINATIONS")
print("-" * 80)

# Create NPI-Year sets from source data (ignoring specialty since we take one per NPI-year)
dme_npi_years = set(
    dme_year_npi[['rfrg_npi', 'year']]
    .dropna()
    .drop_duplicates()
    .apply(tuple, axis=1)
)

gp_npi_years = set(
    gp_providers[['covered_recipient_npi', 'program_year']]
    .dropna()
    .drop_duplicates()
    .apply(lambda x: (x['covered_recipient_npi'], x['program_year']), axis=1)
)

final_npi_years = set(
    one_specialty_per_npi_final[['rfrg_npi', 'year']]
    .dropna()
    .apply(tuple, axis=1)
)

all_source_npi_years = dme_npi_years.union(gp_npi_years)
npi_years_in_final = all_source_npi_years.intersection(final_npi_years)
npi_years_missing = all_source_npi_years - final_npi_years

print(f"Total unique NPI-Year combinations in sources: {len(all_source_npi_years):,}")
print(f"NPI-Year combinations found in final dataset: {len(npi_years_in_final):,}")
print(f"NPI-Year combinations missing from final: {len(npi_years_missing):,}")
print(f"Coverage: {100 * len(npi_years_in_final) / len(all_source_npi_years):.2f}%")

if len(npi_years_missing) > 0:
    print(f"\nWARNING: {len(npi_years_missing)} NPI-Year combinations are missing!")
    print(f"Sample missing combinations (NPI, Year): {list(npi_years_missing)[:5]}")
else:
    print("\nAll NPI-Year combinations are present in final dataset")


5. CHECKING NPI-YEAR COMBINATIONS
--------------------------------------------------------------------------------
Total unique NPI-Year combinations in sources: 1,265,154
NPI-Year combinations found in final dataset: 1,265,154
NPI-Year combinations missing from final: 0
Coverage: 100.00%

All NPI-Year combinations are present in final dataset


In [47]:
one_specialty_per_npi[one_specialty_per_npi["rfrg_npi"] == 1861840316]

,rfrg_npi,rfrg_prvdr_spclty_desc,rfrg_prvdr_spclty_srce,year
635,1861840316,Student in an Organized Health Care Education/...,NPPES-Taxonomy,2023
578310,1861840316,Hospitalist,Claim-Specialty,2021
986449,1861840316,Hospitalist,Claim-Specialty,2022


In [48]:
dme_year_npi[["rfrg_npi","rfrg_prvdr_spclty_desc", "rfrg_prvdr_spclty_srce", "year"]][dme_year_npi['rfrg_npi'] == 1861840316]

,rfrg_npi,rfrg_prvdr_spclty_desc,rfrg_prvdr_spclty_srce,year
333818,1861840316,Hospitalist,Claim-Specialty,2021
719321,1861840316,Hospitalist,Claim-Specialty,2022
1106305,1861840316,Student in an Organized Health Care Education/...,NPPES-Taxonomy,2023


In [49]:
one_specialty_per_npi_final.to_csv("/dsa/groups/casestudycf25/team02/silver/one_specialty_per_npi_year.csv")